In [1]:
import json
import warnings
from pathlib import Path
import platform
import sklearn
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score, precision_score, recall_score, f1_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42
ROOT = Path.cwd().parent 
DATA_PATH = ROOT / "data" / "Customer-Churn-Records.csv"
ARTIFACTS = ROOT / "src" / "churn" / "artifact"


In [2]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(10000, 18)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RowNumber           10000 non-null  int64  
 1   CustomerId          10000 non-null  int64  
 2   Surname             10000 non-null  str    
 3   CreditScore         10000 non-null  int64  
 4   Geography           10000 non-null  str    
 5   Gender              10000 non-null  str    
 6   Age                 10000 non-null  int64  
 7   Tenure              10000 non-null  int64  
 8   Balance             10000 non-null  float64
 9   NumOfProducts       10000 non-null  int64  
 10  HasCrCard           10000 non-null  int64  
 11  IsActiveMember      10000 non-null  int64  
 12  EstimatedSalary     10000 non-null  float64
 13  Exited              10000 non-null  int64  
 14  Complain            10000 non-null  int64  
 15  Satisfaction Score  10000 non-null  int64  
 16  Card Type       

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,5.000500e+03,2886.895680,1.00,2500.75,5.000500e+03,7.500250e+03,10000.00
CustomerId,10000.0,1.569094e+07,71936.186123,15565701.00,15628528.25,1.569074e+07,1.575323e+07,15815690.00
CreditScore,10000.0,6.505288e+02,96.653299,350.00,584.00,6.520000e+02,7.180000e+02,850.00
Age,10000.0,3.892180e+01,10.487806,18.00,32.00,3.700000e+01,4.400000e+01,92.00
Tenure,10000.0,5.012800e+00,2.892174,0.00,3.00,5.000000e+00,7.000000e+00,10.00
Balance,10000.0,7.648589e+04,62397.405202,0.00,0.00,9.719854e+04,1.276442e+05,250898.09
NumOfProducts,10000.0,1.530200e+00,0.581654,1.00,1.00,1.000000e+00,2.000000e+00,4.00
HasCrCard,10000.0,7.055000e-01,0.455840,0.00,0.00,1.000000e+00,1.000000e+00,1.00
IsActiveMember,10000.0,5.151000e-01,0.499797,0.00,0.00,1.000000e+00,1.000000e+00,1.00
EstimatedSalary,10000.0,1.000902e+05,57510.492818,11.58,51002.11,1.001939e+05,1.493882e+05,199992.48


In [8]:
df.columns = (df.columns.str.strip()
                         .str.replace(" ", "_")
                         .str.replace(r"(?<=[a-z])(?=[A-Z])", "_", regex=True)
                         .str.lower())
df.columns.tolist()

['row_number',
 'customer_id',
 'surname',
 'credit_score',
 'geography',
 'gender',
 'age',
 'tenure',
 'balance',
 'num_of_products',
 'has_cr_card',
 'is_active_member',
 'estimated_salary',
 'exited',
 'complain',
 'satisfaction_score',
 'card_type',
 'point_earned']

In [9]:
TARGET = "exited"
rate = df[TARGET].mean()
print(f"Ушедших клиентов: {df[TARGET].sum()} из {len(df)} ({rate:.1%})")
df[TARGET].value_counts()

Ушедших клиентов: 2038 из 10000 (20.4%)


exited
0    7962
1    2038
Name: count, dtype: int64

In [10]:
print("пропусков всего:", int(df.isna().sum().sum()))
print("дубликатов:", int(df.duplicated().sum()))
print("уникальных customer_id:", df["customer_id"].nunique())

пропусков всего: 0
дубликатов: 0
уникальных customer_id: 10000


In [11]:
print(f"complain совпадает с exited в {(df['complain'] == df[TARGET]).mean():.2%} строк")
pd.crosstab(df["complain"], df[TARGET])

complain совпадает с exited в 99.86% строк


exited,0,1
complain,,
0,7952,4
1,10,2034


In [12]:
DROP = ["row_number", "customer_id", "surname", "complain", TARGET]
X = df.drop(columns=DROP)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print(f"доля оттока: train {y_train.mean():.3f}, test {y_test.mean():.3f}")

(8000, 13) (2000, 13)
доля оттока: train 0.204, test 0.204


In [13]:
categorical_cols = ["geography", "gender", "card_type"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]
print("числовые:", numeric_cols)
print("категориальные:", categorical_cols)

числовые: ['credit_score', 'age', 'tenure', 'balance', 'num_of_products', 'has_cr_card', 'is_active_member', 'estimated_salary', 'satisfaction_score', 'point_earned']
категориальные: ['geography', 'gender', 'card_type']


In [14]:
X[numeric_cols].describe().T[["min", "max"]]

,min,max
credit_score,350.00,850.00
age,18.00,92.00
tenure,0.00,10.00
balance,0.00,250898.09
num_of_products,1.00,4.00
has_cr_card,0.00,1.00
is_active_member,0.00,1.00
estimated_salary,11.58,199992.48
satisfaction_score,1.00,5.00
point_earned,119.00,1000.00


In [15]:
{col: sorted(X[col].unique()) for col in categorical_cols}

{'geography': ['France', 'Germany', 'Spain'],
 'gender': ['Female', 'Male'],
 'card_type': ['DIAMOND', 'GOLD', 'PLATINUM', 'SILVER']}

In [16]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

logreg = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)

def report(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "model": name,
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "threshold": threshold,
    }

proba_lr = logreg.predict_proba(X_test)[:, 1]
results = [report("logreg", y_test, proba_lr)]
pd.DataFrame(results).round(3)

,model,ROC-AUC,PR-AUC,precision,recall,F1,threshold
0,logreg,0.778,0.486,0.623,0.211,0.315,0.5


In [21]:
TARGET_RECALL = 0.70

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proba = cross_val_predict(logreg, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)
best_threshold = float(thresholds[recall[:-1] >= TARGET_RECALL].max())
print(f"порог по train (OOF): {best_threshold:.3f}")

results.append(report("logreg @recall>=0.7", y_test, proba_lr, threshold=round(best_threshold, 4)))
pd.DataFrame(results).round(3)

порог по train (OOF): 0.197


,model,ROC-AUC,PR-AUC,precision,recall,F1,threshold
0,logreg,0.778,0.486,0.623,0.211,0.315,0.500
1,logreg @recall>=0.7,0.778,0.486,0.379,0.740,0.502,0.197
2,logreg @recall>=0.7,0.778,0.486,0.379,0.740,0.502,0.197


In [22]:
metrics_test = {k: round(float(v), 4) for k, v in results[-1].items() if k != "model"}

metadata = {
    "model_name": "bank-churn-logreg",
    "model_version": "1.0.0",
    "trained_at": pd.Timestamp.utcnow().isoformat(timespec="seconds"),
    "target": TARGET,
    "n_train": int(len(X_train)),
    "features": list(X.columns),
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "dropped_cols": [c for c in DROP if c != TARGET],
    "threshold": round(best_threshold, 4),
    "threshold_rule": f"max threshold with recall >= {TARGET_RECALL} on train OOF",
    "metrics_test": metrics_test,
    "libs": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

joblib.dump({"pipeline": logreg, "metadata": metadata}, ARTIFACTS / "model.joblib")
(ARTIFACTS / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(metadata, ensure_ascii=False, indent=2))
sorted(p.name for p in ARTIFACTS.iterdir())

{
  "model_name": "bank-churn-logreg",
  "model_version": "1.0.0",
  "trained_at": "2026-09-17T15:56:55+00:00",
  "target": "exited",
  "n_train": 8000,
  "features": [
    "credit_score",
    "geography",
    "gender",
    "age",
    "tenure",
    "balance",
    "num_of_products",
    "has_cr_card",
    "is_active_member",
    "estimated_salary",
    "satisfaction_score",
    "card_type",
    "point_earned"
  ],
  "numeric_cols": [
    "credit_score",
    "age",
    "tenure",
    "balance",
    "num_of_products",
    "has_cr_card",
    "is_active_member",
    "estimated_salary",
    "satisfaction_score",
    "point_earned"
  ],
  "categorical_cols": [
    "geography",
    "gender",
    "card_type"
  ],
  "dropped_cols": [
    "row_number",
    "customer_id",
    "surname",
    "complain"
  ],
  "threshold": 0.1973,
  "threshold_rule": "max threshold with recall >= 0.7 on train OOF",
  "metrics_test": {
    "ROC-AUC": 0.7784,
    "PR-AUC": 0.4865,
    "precision": 0.3794,
    "recall":

['metadata.json', 'model.joblib']